In [5]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from lob_forecasting.data import get_fi2010_split_paths, load_fi2010_file

train_file, test_file = get_fi2010_split_paths(
    project_root=PROJECT_ROOT,
    market="NoAuction",
    normalization="Zscore",
    cf=1,
)

## FI-2010 data layout

FI-2010 files are stored as variables x samples. We verify the expected files are present, then load one training split to inspect the raw shape before transposing to samples x variables.

In [6]:
for path in [train_file, test_file]:
    print(f"{path}: {path.exists()}")

if not train_file.exists():
    raise FileNotFoundError(f"Missing training file: {train_file}")

/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/data/raw/fi2010/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt: True
/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/data/raw/fi2010/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Testing/Test_Dst_NoAuction_ZScore_CF_1.txt: True


In [7]:
raw_train = np.loadtxt(train_file)
print("Raw training shape:", raw_train.shape)

X_train, y_train_all = load_fi2010_file(train_file)

print("X_train:", X_train.shape)
print("y_train_all:", y_train_all.shape)

Raw training shape: (149, 39512)
X_train: (39512, 144)
y_train_all: (39512, 5)


In [ ]:
def summarize_labels(y, split_name):
    print(f"{split_name} label distributions")
    for col in range(y.shape[1]):
        values, counts = np.unique(y[:, col].astype(int), return_counts=True)
        total = counts.sum()
        parts = ", ".join(
            [f"class {value}: {count} ({count / total:.2%})" for value, count in zip(values, counts)]
        )
        print(f"Horizon {col + 1}: {parts}")
    print()

summarize_labels(y_train_all, "Train")

Train label distributions
Horizon 1: class 1: 7476 (18.92%), class 2: 24940 (63.12%), class 3: 7096 (17.96%)
Horizon 2: class 1: 9721 (24.60%), class 2: 20620 (52.19%), class 3: 9171 (23.21%)
Horizon 3: class 1: 11311 (28.63%), class 2: 17712 (44.83%), class 3: 10489 (26.55%)
Horizon 4: class 1: 13349 (33.78%), class 2: 13821 (34.98%), class 3: 12342 (31.24%)
Horizon 5: class 1: 16023 (40.55%), class 2: 8875 (22.46%), class 3: 14614 (36.99%)



## Horizon selection note

For initial modeling in Notebook 02, horizon 3 is used because its class distribution is relatively balanced compared with the other horizons on CF_1.